<div dir="rtl" style="text-align: right">

‏# 07 — افزونه عملی تشخیص ناهنجاری

</div>

In [ ]:
# Persian text rendering for notebook markdown and plots
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": [
        "Vazirmatn",
        "Vazir",
        "IRANSans",
        "Noto Sans Arabic",
        "Noto Naskh Arabic",
        "DejaVu Sans",
    ],
    "axes.unicode_minus": False,
})

try:
    from IPython.display import HTML, display

    display(HTML('<div dir="rtl" style="text-align: right"></div>'))
except Exception:
    pass

<div dir="rtl" style="text-align: right">

‏## مفهوم: چه چیزی ناهنجاری محسوب می‌شود؟

</div>

In [ ]:
import hashlib, json, os, platform, random, warnings
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
# هشدارهای سازگاری/رابط کاربری روی پیش‌بینی‌ها یا اجرای دفترچه اثر ندارند.
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (balanced_accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, precision_score, recall_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TARGET = "y"
LEAKAGE_COLUMNS = ["duration"]

def project_root():
    '''Return the course root when present, otherwise the notebook directory.'''
    # اگر ریشه دوره در دسترس بود همان را برگردان، وگرنه پوشه دفترچه را.
    return ROOT

def set_seed(seed=SEED):
    '''Seed Python and NumPy RNGs for reproducible notebook runs.'''
    random.seed(seed)
    np.random.seed(seed)

def fast_mode():
    '''Report whether notebooks should use the reduced fast-mode sample.'''
    # برای آزمایش‌های کامل، FAST_MODE=0 را تنظیم کنید؛ حالت لپ‌تاپ به‌صورت پیش‌فرض فعال است.
    return os.getenv("FAST_MODE", "1").lower() not in {"0", "false", "no"}

def bank_data_path():
    '''Locate the bundled Bank Marketing CSV file.'''
    # این دوره با یک مجموعه‌داده محلی همراه است؛ دفترچه‌ها هرگز به شبکه دسترسی ندارند.
    path = project_root() / "data" / "raw" / "bank-full.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"Expected the bundled Bank Marketing data at {path}. "
            "Run the notebook from the course root or place bank-full.csv there."
        )
    return path

def file_sha256(path):
    '''Compute the SHA-256 digest of a local file.'''
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_bank_data(include_duration=False):
    '''Load the Bank Marketing dataset and drop leakage columns by default.'''
    # داده را بارگذاری کن، y را encode کن، و مدت تماس پس از تماس را به‌صورت پیش‌فرض حذف کن.
    frame = pd.read_csv(bank_data_path(), sep=";")
    frame[TARGET] = frame[TARGET].map({"no": 0, "yes": 1}).astype("int8")
    if not include_duration:
        frame = frame.drop(columns=LEAKAGE_COLUMNS)
    return frame

def stratified_sample(frame, n, seed=SEED):
    '''Draw a label-preserving sample from a classified dataset.'''
    if n >= len(frame):
        return frame.copy()
    fractions = frame[TARGET].value_counts(normalize=True)
    counts = (fractions * n).round().astype(int)
    counts.iloc[0] += n - counts.sum()
    parts = [group.sample(n=min(counts.loc[label], len(group)),
                          random_state=seed + int(label))
             for label, group in frame.groupby(TARGET)]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

def make_splits(frame=None, reduced=None):
    '''Create deterministic stratified train, validation, and test splits.'''
    # split قطعی و stratified به نسبت 60/20/20؛ مجموعه آزمون تا دفترچه 09 قفل می‌ماند.
    from sklearn.model_selection import train_test_split
    frame = load_bank_data() if frame is None else frame
    train_val, test = train_test_split(
        frame, test_size=0.20, stratify=frame[TARGET], random_state=SEED)
    train, validation = train_test_split(
        train_val, test_size=0.25, stratify=train_val[TARGET], random_state=SEED)
    reduced = fast_mode() if reduced is None else reduced
    if reduced:
        train = stratified_sample(train, 12_000)
        validation = stratified_sample(validation, 4_000, SEED + 1)
        test = stratified_sample(test, 4_000, SEED + 2)
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

def split_xy(frame):
    '''Separate a frame into feature matrix and target vector.'''
    return frame.drop(columns=TARGET), frame[TARGET]

def feature_groups(frame):
    '''Identify numeric and categorical feature columns.'''
    features = frame.drop(columns=[TARGET], errors="ignore")
    categorical = features.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical = features.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

def make_preprocessor(frame, scale_numeric=True):
    '''Build the preprocessing pipeline for numeric and categorical features.'''
    # preprocessing فقط داخل pipeline آموزش/CV مربوطه fit می‌شود.
    numerical, categorical = feature_groups(frame)
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=10, sparse_output=True)),
    ])
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numerical),
        ("categorical", categorical_pipe, categorical),
    ], sparse_threshold=0.3)

def classification_metrics(y_true, probability, threshold=0.5):
    '''Compute ranking and threshold-based classification metrics.'''
    prediction = np.asarray(probability) >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"log_loss": log_loss(y_true, probability),
            "brier_score": brier_score_loss(y_true, probability),
            "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "cost": float(fp + 5 * fn)}

def threshold_table(y_true, probability, thresholds=None):
    '''Evaluate classification metrics across a list of decision thresholds.'''
    thresholds = np.linspace(0.05, 0.80, 76) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t),
                          **classification_metrics(y_true, probability, float(t))}
                         for t in thresholds])

def add_domain_features(frame):
    '''Create domain-inspired features for the Bank Marketing dataset.'''
    result = frame.copy()
    result["was_previously_contacted"] = (result["pdays"] != -1).astype("int8")
    result["pdays_clean"] = result["pdays"].replace(-1, np.nan)
    result["contact_pressure"] = result["campaign"] / (1 + result["previous"])
    result["balance_per_age"] = result["balance"] / result["age"].clip(lower=18)
    result["age_band"] = pd.cut(result["age"], bins=[0, 29, 39, 49, 59, np.inf],
                                labels=["<30", "30s", "40s", "50s", "60+"]).astype("object")
    return result.drop(columns=["pdays"])

def environment_metadata():
    '''Collect runtime metadata for experiment tracking.'''
    import sklearn
    return {"python": platform.python_version(), "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__}

def write_json(data, path):
    '''Serialize structured data to a JSON file on disk.'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

set_seed(SEED)
FAST_MODE = fast_mode()
CV_FOLDS = 3 if FAST_MODE else 5
sns.set_theme(style="whitegrid", context="notebook", palette=[BRAND_COLOR, ACCENT_COLOR, "#264653", "#f4a261"])
plt.rcParams.update({
    "axes.facecolor": "#fcfcfc",
    "figure.facecolor": "white",
    "axes.edgecolor": "#d9d9d9",
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 30)
print({"FAST_MODE": FAST_MODE, "CV_FOLDS": CV_FOLDS, "seed": SEED})

In [ ]:
import lightgbm as lgb
import shap
from scipy import sparse

development, validation, _sealed_test = make_splits(load_bank_data(), reduced=FAST_MODE)
X_dev, y_dev = split_xy(development)
X_val, y_val = split_xy(validation)
preprocessor = make_preprocessor(development, scale_numeric=False)
X_dev_e = preprocessor.fit_transform(X_dev, y_dev)
X_val_e = preprocessor.transform(X_val)
feature_names = preprocessor.get_feature_names_out()
model = lgb.LGBMClassifier(
    n_estimators=250 if FAST_MODE else 500, learning_rate=.04, num_leaves=31,
    min_child_samples=40, subsample=.9, colsample_bytree=.9,
    random_state=SEED, n_jobs=-1, verbosity=-1,
).fit(X_dev_e, y_dev)
print("SHAP", shap.__version__, "features", len(feature_names))

<div dir="rtl" style="text-align: right">

‏نرخ اشتراک فقط برای جهت‌گیری نشان داده شده است. یک کلاس supervised کمیاب، به‌طور خودکار کلاس ناهنجاری نیست. اگر آن را چنین فرض کنیم، مسئله را اشتباه صورت‌بندی می‌کنیم.

</div>

In [ ]:
sample_n = 300 if FAST_MODE else 1000
explain_rows = X_val.sample(n=min(sample_n, len(X_val)), random_state=SEED)
X_explain = preprocessor.transform(explain_rows)
if sparse.issparse(X_explain):
    X_explain = X_explain.toarray()
X_explain = pd.DataFrame(X_explain, columns=feature_names, index=explain_rows.index)
explainer = shap.TreeExplainer(model)
explanation = explainer(X_explain)
print("explanation shape:", explanation.shape)

<div dir="rtl" style="text-align: right">

‏## Global magnitude and direction

</div>

In [ ]:
shap.plots.bar(explanation, max_display=15, show=False)
plt.title("Global mean |SHAP| on validation sample")
plt.tight_layout(); plt.show()
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align: right">

‏## برچسب‌های اعتبارسنجی کنترل‌شده

</div>

In [ ]:
top_feature = feature_names[np.abs(explanation.values).mean(axis=0).argmax()]
shap.plots.scatter(explanation[:, top_feature], color=explanation, show=False)
plt.title(f"Dependence: {top_feature}")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align: right">

‏## baseline عملی: قوانین ساده کسب‌وکار

</div>

In [ ]:
probabilities = model.predict_proba(X_explain)[:, 1]
local_position = int(np.argmax(probabilities))
print("client row:", X_explain.index[local_position], "probability:", probabilities[local_position])
shap.plots.waterfall(explanation[local_position], max_display=12, show=False)
plt.tight_layout(); plt.show()
shap.plots.force(explanation[local_position], matplotlib=True, show=False)
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align: right">

‏امتیاز قانون 0 یعنی هیچ‌یک از فیلدهای تحت نظر از آستانه خود عبور نکرده‌اند. امتیازهای بالاتر یعنی نقض قانون‌های بیشتر.

</div>

In [ ]:
model_2 = lgb.LGBMClassifier(
    n_estimators=250 if FAST_MODE else 500, learning_rate=.04, num_leaves=31,
    min_child_samples=40, subsample=.85, colsample_bytree=.85,
    random_state=SEED + 1, n_jobs=-1, verbosity=-1,
).fit(X_dev_e, y_dev)
explanation_2 = shap.TreeExplainer(model_2)(X_explain)
importance_1 = pd.Series(np.abs(explanation.values).mean(axis=0), index=feature_names)
importance_2 = pd.Series(np.abs(explanation_2.values).mean(axis=0), index=feature_names)
stability = importance_1.corr(importance_2, method="spearman")
print("Spearman rank stability:", round(stability, 3))
display(pd.DataFrame({"seed_42": importance_1, "seed_43": importance_2})
        .sort_values("seed_42", ascending=False).head(15))

<div dir="rtl" style="text-align: right">

‏Responsible communication reports the prediction moment, background/reference population,
‏model version, feature definitions, uncertainty, known correlations, and a plain warning that
‏attribution is not causation. Explanations can faithfully expose an undesirable model; they do
‏not make that model fair or safe.

‏## Common mistakes and leakage warnings

‏- Using a generic model-agnostic explainer when an efficient exact tree explainer is available.
‏- Explaining training rows and presenting them as general behavior.
‏- Reading SHAP direction as a causal intervention.
‏- Ignoring preprocessing names and reporting transformed columns ambiguously.
‏- Treating one explanation seed/sample as stable truth.

‏## Exercises

‏1. Aggregate one-hot SHAP magnitudes to original features.
‏2. Compare explanations for a false positive and false negative on validation.
‏3. **Challenge:** bootstrap the validation sample 100 times and attach uncertainty intervals to
‏   the top-10 global importances; report rank instability.

‏## Summary

‏SHAP provides model-relative global and local attributions. Correct explainer choice, explicit
‏reference data, transformed-feature provenance, stability checks, and causal humility are part
‏of the method—not optional communication polish.

‏## References

‏- [SHAP TreeExplainer](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html)
‏- [SHAP plots API](https://shap.readthedocs.io/en/latest/api.html#plots)
‏- [Interpretable ML: SHAP chapter](https://christophm.github.io/interpretable-ml-book/shap.html)

</div>